### Start SparkSession

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F


spark = (
    SparkSession
    .builder
    .appName("pyspark_local")
    .getOrCreate()
)

print("Активные Spark сессии:", spark.sparkContext.uiWebUrl)

26/03/31 19:43:20 WARN Utils: Your hostname, user-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/31 19:43:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 19:43:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/31 19:43:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Активные Spark сессии: http://10.0.2.15:4041


### Read

In [2]:
PATH = 'data/price_paid_records.csv'

In [3]:
spark.read.csv(PATH).show(5)

+--------------------+-----+----------------+-------------+-------+--------+----------+------------------+------------------+----------------+--------------------+
|                 _c0|  _c1|             _c2|          _c3|    _c4|     _c5|       _c6|               _c7|               _c8|             _c9|                _c10|
+--------------------+-----+----------------+-------------+-------+--------+----------+------------------+------------------+----------------+--------------------+
|Transaction uniqu...|Price|Date of Transfer|Property Type|Old/New|Duration| Town/City|          District|            County|PPDCategory Type|Record Status - m...|
|{81B82214-7FBC-41...|25000|1995-08-18 00:00|            T|      N|       F|    OLDHAM|            OLDHAM|GREATER MANCHESTER|               A|                   A|
|{8046EC72-1466-42...|42500|1995-08-09 00:00|            S|      N|       F|     GRAYS|          THURROCK|          THURROCK|               A|                   A|
|{278D581A-5BF3-

In [5]:
df = spark.read.csv(PATH, sep=',', header=True)

# df.show(5, truncate=False)

df.show(2, False, True)

-RECORD 0-------------------------------------------------------------------
 Transaction unique identifier     | {81B82214-7FBC-4129-9F6B-4956B4A663AD} 
 Price                             | 25000                                  
 Date of Transfer                  | 1995-08-18 00:00                       
 Property Type                     | T                                      
 Old/New                           | N                                      
 Duration                          | F                                      
 Town/City                         | OLDHAM                                 
 District                          | OLDHAM                                 
 County                            | GREATER MANCHESTER                     
 PPDCategory Type                  | A                                      
 Record Status - monthly file only | A                                      
-RECORD 1-------------------------------------------------------------------

### PrintSchema

In [5]:
df.printSchema()

root
 |-- Transaction unique identifier: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Date of Transfer: string (nullable = true)
 |-- Property Type: string (nullable = true)
 |-- Old/New: string (nullable = true)
 |-- Duration: string (nullable = true)
 |-- Town/City: string (nullable = true)
 |-- District: string (nullable = true)
 |-- County: string (nullable = true)
 |-- PPDCategory Type: string (nullable = true)
 |-- Record Status - monthly file only: string (nullable = true)



### Select

In [15]:
df.select('Town/City').distinct().show(10, truncate=False)

[Stage 18:=================================================>      (16 + 2) / 18]

+-------------------+
|Town/City          |
+-------------------+
|HIGH WYCOMBE       |
|NEWQUAY            |
|ST. LEONARDS-ON-SEA|
|BERWICK-UPON-TWEED |
|GOODWICK           |
|ROTHERHAM          |
|CHOPPINGTON        |
|LEE-ON-THE-SOLENT  |
|STAMFORD           |
|LLANGEFNI          |
+-------------------+
only showing top 10 rows



### GroupBy и OrderBy

In [6]:
df = df.withColumn("Price", F.col("Price").cast('int'))
(
    df
    .groupBy('Town/City')
    .agg(F.avg('Price').alias('Price'))
    .withColumn('Price', F.round(F.col('Price'), 0))
    .orderBy(F.col('Price'))
    
).show()

[Stage 4:==================================================>      (16 + 2) / 18]

+--------------------+-------+
|           Town/City|  Price|
+--------------------+-------+
|              WARLEY|30307.0|
|               KELSO|36000.0|
|            FERNDALE|41622.0|
|        NEW TREDEGAR|50235.0|
|            TREORCHY|51640.0|
|              PENTRE|54050.0|
|         ABERTILLERY|55212.0|
|           TONYPANDY|56188.0|
|        MOUNTAIN ASH|57252.0|
|           FERRYHILL|62432.0|
|             SHILDON|62474.0|
|              NELSON|64789.0|
|  BLAENAU FFESTINIOG|65830.0|
|             BARGOED|66653.0|
|NEWBIGGIN-BY-THE-SEA|67251.0|
|            TREDEGAR|67820.0|
|              BOOTLE|68390.0|
|            PETERLEE|69791.0|
|     TRIMDON STATION|71248.0|
|          ACCRINGTON|72580.0|
+--------------------+-------+
only showing top 20 rows



### Filter

In [7]:
df_de = (
    df
    .where(F.col('Town/City') == 'WARLEY')
    .where(F.col('Property Type').isNotNull())
)
df_de.show(2, False, True)

[Stage 7:>                                                          (0 + 1) / 1]

-RECORD 0-------------------------------------------------------------------
 Transaction unique identifier     | {AD77E985-D565-4AD7-B840-2B3CD1F6E325} 
 Price                             | 29000                                  
 Date of Transfer                  | 1995-08-14 00:00                       
 Property Type                     | T                                      
 Old/New                           | N                                      
 Duration                          | F                                      
 Town/City                         | WARLEY                                 
 District                          | SANDWELL                               
 County                            | WEST MIDLANDS                          
 PPDCategory Type                  | A                                      
 Record Status - monthly file only | A                                      
-RECORD 1-------------------------------------------------------------------

### Save to CSV

In [7]:
df_de.columns

['Transaction unique identifier',
 'Price',
 'Date of Transfer',
 'Property Type',
 'Old/New',
 'Duration',
 'Town/City',
 'District',
 'County',
 'PPDCategory Type',
 'Record Status - monthly file only']

In [8]:
final = (
    df_de
    .select(
        'Transaction unique identifier',
        F.col('Price').cast('int'),
        F.col('Date of Transfer').cast('date'),
        'Property Type',
        'Old/New',
        'Town/City',
        'District',
        'County',
        'PPDCategory Type',
    )
)

final.show(2, False, True)

[Stage 8:>                                                          (0 + 1) / 1]

-RECORD 0---------------------------------------------------------------
 Transaction unique identifier | {AD77E985-D565-4AD7-B840-2B3CD1F6E325} 
 Price                         | 29000                                  
 Date of Transfer              | 1995-08-14                             
 Property Type                 | T                                      
 Old/New                       | N                                      
 Town/City                     | WARLEY                                 
 District                      | SANDWELL                               
 County                        | WEST MIDLANDS                          
 PPDCategory Type              | A                                      
-RECORD 1---------------------------------------------------------------
 Transaction unique identifier | {A235691B-413F-4E42-A5B9-D726AA6E82D5} 
 Price                         | 29500                                  
 Date of Transfer              | 1995-08-14        

### Разное число партиций

In [9]:
# Сохранение неконтроллируемое по кол-ву файлов
(
    final
    .write.mode("overwrite")
    .format('csv')
    .options(header='True', sep=';')
    .csv('data/final_no_control')
)

partition_num = final.rdd.getNumPartitions()
print(f'Кол-во партиций {partition_num}')

# Сохранение контроллируемое по кол-ву файлов - ОДИН ФАЙЛ
(
    final
    .coalesce(1)
    .write.mode("overwrite")
    .format('csv')
    .options(header='True', sep=';')
    .csv('data/final_one_file')
)

partition_num = final.coalesce(1).rdd.getNumPartitions()
print(f'Кол-во партиций {partition_num}')

Кол-во партиций 18


[Stage 10:>                                                         (0 + 1) / 1]

Кол-во партиций 1


### Партицирование по колонке

In [12]:
# Сохранения с партицированием
(
    final
    .write.mode("overwrite")
    .partitionBy('Date of Transfer')
    .format('csv')
    .options(header='True', sep=';')
    .csv('data/final_partitioned')
)

print_df = final.select('Date of Transfer').distinct()
print(f'Кол-во партиций: {print_df.count()}')


# Сохранения с партицированием и repartition внутри самой партиции
(
    final
    .repartition(1, 'County')
    .write.mode("overwrite")
    .partitionBy('Date of Transfer')
    .format('csv')
    .options(header='True', sep=';')
    .csv('data/final_partitioned_repart')
)

partition_num = final.repartition(1, 'Date of Transfer').rdd.getNumPartitions()
print(f'Кол-во партиций c repartition(1, \'County\') {partition_num}')

Кол-во партиций: 25


[Stage 48:====================================================>   (17 + 1) / 18]

Кол-во партиций c repartition(1, 'County') 1


### Read Transformed

In [11]:
reader_no_control = (
    spark
    .read
    .csv('data/final_no_control/', header=True, sep=';')
    .where(''' County = "WEST MIDLANDS" ''')
)

reader_final_one_file = (
    spark
    .read
    .csv('data/final_one_file/', header=True, sep=';')
    .where(''' County = "WEST MIDLANDS" ''')
)

reader_partitioned = (
    spark
    .read
    .csv('data/final_partitioned', header=True, sep=';')
    .where(''' County = "WEST MIDLANDS" ''')
)

reader_partitioned_repart = (
    spark
    .read
    .csv('data/final_partitioned_repart', header=True, sep=';')
    .where(''' County = "WEST MIDLANDS" ''')
)


reader_no_control.count() 

reader_final_one_file.count() 

reader_partitioned.count() 

reader_partitioned_repart.count() 

28

### JOIN

In [13]:
data = [
    ('{AD77E985-D565-4AD7-B840-200000000000}', "PENTRE"),
    ('{AD77E985-D565-4AD7-B840-200000000001}', "TONYPANDY"),
    ('{AD77E985-D565-4AD7-B840-200000000002}', "BOOTLE"),
    ('{AD77E985-D565-4AD7-B840-200000000003}', "SHILDON"),
    ('{AD77E985-D565-4AD7-B840-200000000004}', "ACCRINGTON"),
]

region_df = spark.createDataFrame(data, schema='identifier string, Town_City string')

region_df.show()


customs_data = (
    spark
    .read
    .csv(PATH, header=True, sep=',')
)

customs_data.show(2, False, True)

+--------------------+----------+
|          identifier| Town_City|
+--------------------+----------+
|{AD77E985-D565-4A...|    PENTRE|
|{AD77E985-D565-4A...| TONYPANDY|
|{AD77E985-D565-4A...|    BOOTLE|
|{AD77E985-D565-4A...|   SHILDON|
|{AD77E985-D565-4A...|ACCRINGTON|
+--------------------+----------+

-RECORD 0-------------------------------------------------------------------
 Transaction unique identifier     | {81B82214-7FBC-4129-9F6B-4956B4A663AD} 
 Price                             | 25000                                  
 Date of Transfer                  | 1995-08-18 00:00                       
 Property Type                     | T                                      
 Old/New                           | N                                      
 Duration                          | F                                      
 Town/City                         | OLDHAM                                 
 District                          | OLDHAM                                 


In [14]:
# Отключим автоматический Broadcast JOIN
import time
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [15]:
# Замерим выполнение запроса без broadcast join

start_time = time.time()

customs_data.join(region_df, customs_data['Transaction unique identifier']==region_df.identifier, "left").count()

end_time = time.time()

print(f"Время выполнения join операции: {end_time - start_time:.2f} секунд")

[Stage 57:===================================================>    (12 + 1) / 13]

Время выполнения join операции: 126.51 секунд


In [16]:
# Замерим выполнение запроса c broadcast join

start_time = time.time()

customs_data.join(F.broadcast(region_df), customs_data['Transaction unique identifier'] == region_df.identifier, "left").count()

end_time = time.time()

print(f"Время выполнения c broadcast join: {end_time - start_time:.2f} секунд")

[Stage 63:====================================================>   (17 + 1) / 18]

Время выполнения c broadcast join: 70.05 секунд


### Repartition и Coalesce

In [32]:
data = [
    (1,'one'),
    (2,'two'),
    (3,'three'),
    (4,'four'),
    (5,'five'),
    (6,'six'),
    (7, 'seven'),
    (8, 'eight'),
    (9, 'nine'),
]

df = spark.createDataFrame(data, ['id', 'number'])

df.show()

+---+------+
| id|number|
+---+------+
|  1|   one|
|  2|   two|
|  3| three|
|  4|  four|
|  5|  five|
|  6|   six|
|  7| seven|
|  8| eight|
|  9|  nine|
+---+------+



In [33]:
# Намеренно перемешаем и поделим на 8 разделов
mix = df.repartition(8)
mix.rdd.glom().collect()

[[Row(id=8, number='eight')],
 [Row(id=9, number='nine')],
 [Row(id=1, number='one'),
  Row(id=4, number='four'),
  Row(id=7, number='seven')],
 [Row(id=3, number='three'), Row(id=6, number='six')],
 [Row(id=2, number='two'), Row(id=5, number='five')],
 [],
 [],
 []]

In [34]:
mix.repartition(3).rdd.glom().collect()

[[Row(id=1, number='one'), Row(id=5, number='five'), Row(id=9, number='nine')],
 [Row(id=3, number='three'),
  Row(id=4, number='four'),
  Row(id=7, number='seven')],
 [Row(id=2, number='two'), Row(id=6, number='six'), Row(id=8, number='eight')]]

In [35]:
mix.coalesce(3).rdd.glom().collect()

[[Row(id=8, number='eight'),
  Row(id=3, number='three'),
  Row(id=6, number='six')],
 [Row(id=9, number='nine'), Row(id=2, number='two'), Row(id=5, number='five')],
 [Row(id=1, number='one'),
  Row(id=4, number='four'),
  Row(id=7, number='seven')]]

In [37]:
spark.stop()